In [2]:
#aggregates functions(SUM,AVG,COUNT) per complex queries

In [3]:
#01:Multiple aggregates ek saath, JOIN ke sath combine:

In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

q1 = pd.read_sql("""
    SELECT c.Country,
            COUNT(DISTINCT c."Customer ID") as total_customers,
            COUNT(DISTINCT o.Invoice) as total_orders,
            SUM(o.OrderValue) as total_revenue,
            AVG(o.OrderValue) as avg_order_value,
            MAX(o.OrderValue) as max_order_value,
            MIN(o.OrderValue) as min_order_value
    FROM customers c
    INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
    GROUP BY c.Country
    ORDER BY total_revenue DESC
""", conn)

print(q1)

                 Country  total_customers  total_orders  total_revenue  \
0         United Kingdom             5410         40506   1.380662e+07   
1                   EIRE                5           729   5.792255e+05   
2            Netherlands               23           250   5.485249e+05   
3                Germany              107          1097   4.189031e+05   
4                 France               95           740   3.268431e+05   
5              Australia               15           134   1.722312e+05   
6                  Spain               41           224   1.037355e+05   
7            Switzerland               22           133   1.010288e+05   
8                 Sweden               19           128   8.745542e+04   
9                Belgium               29           216   7.415863e+04   
10               Denmark               12            69   7.294083e+04   
11              Portugal               24           122   5.240545e+04   
12                 Japan              

In [5]:
#02:Nested aggregation- subquery ke andar aggregate, phir uspar aggregate:

In [6]:
#pehle har customer ka total spend nikalo, phir un totals ka average nikalo (average customer lifetime value jaisa)
q2 = pd.read_sql("""
    SELECT AVG(customer_total) as avg_customer_value
    FROM (
        SELECT "Customer ID", SUM(Quantity * Price) as customer_total
        FROM transactions
        WHERE "Customer ID" IS NOT NULL AND Quantity > 0
        GROUP BY "Customer ID"
    )
""",conn)

print(q2)

   avg_customer_value
0         3017.076888


In [7]:
#03:Conditional aggregation- CASE WHEN ke saath(bahut important pattern, RFM/churn analysis me kaam aaiga):

In [8]:
# Har country ke liye: kitne "high value" orders (>100) vs "low value" orders (<=100)
q3 = pd.read_sql("""
    SELECT Country,
           SUM(CASE WHEN (Quantity * Price) > 100 THEN 1 ELSE 0 END) as high_value_orders,
           SUM(CASE WHEN (Quantity * Price) <= 100 THEN 1 ELSE 0 END) as low_value_orders,
           COUNT(*) as total_line_items
    FROM transactions
    WHERE Quantity > 0
    GROUP BY Country
    ORDER BY high_value_orders DESC
    LIMIT 10
""", conn)
print(q3)

          Country  high_value_orders  low_value_orders  total_line_items
0  United Kingdom              21046            940178            961224
1     Netherlands               2102              2991              5093
2            EIRE               1070             16284             17354
3       Australia                524              1291              1815
4         Germany                495             16208             16703
5          Sweden                327              1011              1338
6          France                315             13626             13941
7     Switzerland                186              2951              3137
8           Japan                147               338               485
9           Spain                134              3586              3720


In [9]:
#04:Percentage calculation using aggregates (business insight style):

In [10]:
q4 = pd.read_sql("""
    SELECT Country,
           COUNT(DISTINCT "Customer ID") as customers,
           ROUND(COUNT(DISTINCT "Customer ID") * 100.0 / (SELECT COUNT(DISTINCT "Customer ID") FROM customers), 2) as pct_of_total_customers
    FROM customers
    GROUP BY Country
    ORDER BY customers DESC
    LIMIT 10
""", conn)
print(q4)

          Country  customers  pct_of_total_customers
0  United Kingdom       5410                   91.05
1         Germany        107                    1.80
2          France         95                    1.60
3           Spain         41                    0.69
4         Belgium         29                    0.49
5        Portugal         24                    0.40
6     Netherlands         23                    0.39
7     Switzerland         22                    0.37
8          Sweden         19                    0.32
9           Italy         17                    0.29


In [11]:
#05:Monthly aggregation(date se month nikal kar group karna - Recently/trend analysis ke liye jaruri pattern):

In [12]:
q5 = pd.read_sql("""
    SELECT strftime('%Y-%m', InvoiceDate) as year_month,
           COUNT(DISTINCT Invoice) as total_orders,
           SUM(Quantity * Price) as total_revenue
    FROM transactions
    WHERE Quantity > 0
    GROUP BY year_month
    ORDER BY year_month
""", conn)
print(q5)

   year_month  total_orders  total_revenue
0     2009-12          1839     825685.760
1     2010-01          1205     652708.502
2     2010-02          1282     553713.306
3     2010-03          1770     833570.131
4     2010-04          1513     627934.632
5     2010-05          1642     659858.860
6     2010-06          1719     752270.140
7     2010-07          1585     606681.150
8     2010-08          1511     697274.910
9     2010-09          1911     924333.011
10    2010-10          2371    1126558.040
11    2010-11          2868    1470272.482
12    2010-12          1629    1262598.790
13    2011-01          1120     691364.560
14    2011-02          1126     523631.890
15    2011-03          1531     717639.360
16    2011-04          1318     537808.621
17    2011-05          1731     770536.020
18    2011-06          1576     761739.900
19    2011-07          1540     719221.191
20    2011-08          1409     737014.260
21    2011-09          1896    1058590.172
22    2011-

In [20]:
#practice questions

In [14]:
#01:Har country ka average order value nikaalo, sirf un countries ke liye jinke 5 se zyada customers hain (HAVING use karo).

In [15]:
#

In [16]:
#02:CASE WHEN use karke customers ko 3 buckets mein baanto: "Low spender" (<500), "Medium spender" (500-2000), "High spender" (>2000) — aur har bucket mein kitne customers hain count karo.

In [17]:
#

In [18]:
#03:Kaunsa month sabse zyada revenue wala tha?

In [19]:
#

In [21]:
conn.close()